# 04 - Zillow Data Pipeline

Loads Zillow ZHVI county-level data, assigns Karl & Koss climate regions, and computes
region-minus-affected baselines for each county-month storm observation.

**Inputs:**
- Zillow ZHVI county CSV (fetched directly)
- `../data/processed/storm_events.pkl` — to identify affected counties per month

**Output:** `../data/processed/zillow_panel.pkl` — one row per county-month storm observation
with ZHVI indexed to 100 at storm month and regional baseline ZHVI for comparison

**Baseline definition:** Equal-weighted mean ZHVI of all counties in the same Karl & Koss
climate region that had no recorded storm event in that month.

**Karl & Koss climate regions (1984):**
Nine climatologically-defined regions used throughout NOAA climate research.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

## Karl & Koss Climate Region Lookup
State-level assignment per Karl, T.R. and Koss, W.J. (1984).

In [2]:
# Keyed by 2-digit state FIPS string
KARL_KOSS_REGIONS = {
    # Northeast: CT, DE, ME, MD, MA, NH, NJ, NY, PA, RI, VT
    '09': 'Northeast', '10': 'Northeast', '23': 'Northeast', '24': 'Northeast',
    '25': 'Northeast', '33': 'Northeast', '34': 'Northeast', '36': 'Northeast',
    '42': 'Northeast', '44': 'Northeast', '50': 'Northeast',
    # East North Central: IA, MI, MN, WI  (note: Karl & Koss excludes IL, IN, OH)
    '19': 'East North Central', '26': 'East North Central',
    '27': 'East North Central', '55': 'East North Central',
    # Central: IL, IN, KY, MO, OH, TN, WV
    '17': 'Central', '18': 'Central', '21': 'Central', '29': 'Central',
    '39': 'Central', '47': 'Central', '54': 'Central',
    # Southeast: AL, FL, GA, NC, SC, VA
    '01': 'Southeast', '12': 'Southeast', '13': 'Southeast',
    '37': 'Southeast', '45': 'Southeast', '51': 'Southeast',
    # South: AR, LA, MS, OK, TX
    '05': 'South', '22': 'South', '28': 'South', '40': 'South', '48': 'South',
    # West North Central: KS, MN, MO, NE, ND, SD  (MN/MO split across regions in K&K)
    '20': 'West North Central', '31': 'West North Central',
    '38': 'West North Central', '46': 'West North Central',
    # Southwest: AZ, CO, NM, UT
    '04': 'Southwest', '08': 'Southwest', '35': 'Southwest', '49': 'Southwest',
    # Northwest: ID, MT, OR, WA, WY
    '16': 'Northwest', '30': 'Northwest', '41': 'Northwest',
    '53': 'Northwest', '56': 'Northwest',
    # West: AK, CA, HI, NV
    '02': 'West', '06': 'West', '15': 'West', '32': 'West',
    # DC
    '11': 'Northeast',
}

## ZillowDataParser
Unchanged from `NOAA_Storm_and_Zillow_Data_Pipeline.ipynb`

In [3]:
class ZillowDataParser():
    DTYPES = {
        'RegionID': 'Int64',
        'SizeRank': 'Int64',
        'RegionName': 'object',
        'RegionType': 'object',
        'StateName': 'object',
        'State': 'object',
        'Metro': 'object',
        'StateCodeFIPS': 'Int64',
        'MunicipalCodeFIPS': 'Int64'
    }
    
    META_DATA_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
                      'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS',
                      'latitude', 'longitude']
    
    def __init__(self, regional_url, national_url):
        self.download_data(regional_url, national_url)
        
    def download_data(self, regional_url, national_url):
        regional_data_raw = pd.read_csv(regional_url)
        national_data_raw = pd.read_csv(national_url)
        national = national_data_raw.loc[national_data_raw["RegionName"] == "United States"]
        self.df = pd.concat([regional_data_raw, national], sort=False, ignore_index=True)
        self.df = self.df.astype(self.DTYPES)
        all_columns = self.df.columns.tolist() + [col for col in self.META_DATA_COLS if col not in self.df.columns]
        self.df = self.df.reindex(columns=all_columns)

    def date_cols(self):
        dates = [col for col in self.df.columns if col not in self.META_DATA_COLS]
        dates = sorted(dates, key=pd.to_datetime)
        return list(dates)

    def get_monthly_panel(self):
        date_cols = self.date_cols()
        long_df = self.df.melt(
            id_vars=self.META_DATA_COLS,
            value_vars=date_cols,
            var_name='date',
            value_name='zhvi'
        )
        long_df['date'] = pd.to_datetime(long_df['date'])
        return long_df

## Load Zillow Data

In [4]:
zillow_county_url = "https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"
zillow_msa_url    = "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"

# Save raw files for reproducibility
import urllib.request
county_raw_path = Path('../data/raw/zillow_county_zhvi.csv')
msa_raw_path    = Path('../data/raw/zillow_msa_zhvi.csv')

if not county_raw_path.exists():
    print('Downloading Zillow county ZHVI...')
    urllib.request.urlretrieve(zillow_county_url, county_raw_path)
else:
    print('Using cached zillow_county_zhvi.csv')

if not msa_raw_path.exists():
    print('Downloading Zillow MSA ZHVI...')
    urllib.request.urlretrieve(zillow_msa_url, msa_raw_path)
else:
    print('Using cached zillow_msa_zhvi.csv')

zillow = ZillowDataParser(str(county_raw_path), str(msa_raw_path))
print(f'Loaded {len(zillow.df):,} rows, {len(zillow.date_cols())} months')

Using cached zillow_county_zhvi.csv
Using cached zillow_msa_zhvi.csv
Loaded 3,074 rows, 314 months


## Build Long Panel and Assign Climate Regions

In [5]:
panel = zillow.get_monthly_panel()

# Keep only county-level rows (drop national, MSA etc)
panel = panel[panel['RegionType'] == 'county'].copy()

# Build stcofips
panel['state_fips']  = panel['StateCodeFIPS'].astype(str).str.zfill(2)
panel['county_fips'] = panel['MunicipalCodeFIPS'].astype(str).str.zfill(3)
panel['stcofips']    = panel['state_fips'] + panel['county_fips']

# Assign Karl & Koss climate region
panel['climate_region'] = panel['state_fips'].map(KARL_KOSS_REGIONS)

# Extract year and month
panel['year']  = panel['date'].dt.year
panel['month'] = panel['date'].dt.month

# Filter to storm event years only
panel = panel[panel['year'].between(2020, 2025)].copy()

# Drop rows with no ZHVI or no region assignment
n_before = len(panel)
panel = panel.dropna(subset=['zhvi', 'climate_region'])
print(f'Dropped {n_before - len(panel):,} rows with missing ZHVI or unassigned region')
print(f'Panel shape: {panel.shape}')
print(f'Climate regions: {sorted(panel["climate_region"].unique())}')

Dropped 1,695 rows with missing ZHVI or unassigned region
Panel shape: (219561, 19)
Climate regions: ['Central', 'East North Central', 'Northeast', 'Northwest', 'South', 'Southeast', 'Southwest', 'West', 'West North Central']


## Load Storm Events and Compute Region-Minus-Affected Baseline

In [6]:
storms = pd.read_pickle('../data/processed/storm_events.pkl')

# Set of affected county-month pairs
affected = set(zip(storms['stcofips'], storms['year'], storms['month']))
print(f'Affected county-month observations: {len(affected):,}')

# Flag affected counties in the full panel
panel['affected'] = panel.apply(
    lambda r: (r['stcofips'], r['year'], r['month']) in affected, axis=1
)

print(f'Affected rows in Zillow panel: {panel["affected"].sum():,}')
print(f'Unaffected rows (baseline pool): {(~panel["affected"]).sum():,}')

Affected county-month observations: 14,660
Affected rows in Zillow panel: 14,166
Unaffected rows (baseline pool): 205,395


In [7]:
# Compute regional baseline: equal-weighted mean ZHVI of unaffected counties
# per climate_region x year x month
# Primary baseline: unaffected counties only
baseline_unaffected = (
    panel[~panel['affected']]
    .groupby(['climate_region', 'year', 'month'])['zhvi']
    .agg(baseline_zhvi='mean', baseline_n_counties='count')
    .reset_index()
)

# Fallback baseline: full region mean for months where no unaffected counties exist
baseline_full = (
    panel
    .groupby(['climate_region', 'year', 'month'])['zhvi']
    .agg(baseline_zhvi='mean', baseline_n_counties='count')
    .reset_index()
    .rename(columns={'baseline_zhvi': 'baseline_zhvi_full', 
                     'baseline_n_counties': 'baseline_n_counties_full'})
)

# Merge and fill gaps
baseline = baseline_unaffected.merge(baseline_full, on=['climate_region', 'year', 'month'], how='right')
baseline['baseline_zhvi']      = baseline['baseline_zhvi'].fillna(baseline['baseline_zhvi_full'])
baseline['baseline_n_counties'] = baseline['baseline_n_counties'].fillna(0)
baseline = baseline.drop(columns=['baseline_zhvi_full', 'baseline_n_counties_full'])
baseline.head()

,climate_region,year,month,baseline_zhvi,baseline_n_counties
0,Central,2020,1,137612.092621,657
1,Central,2020,2,138278.976914,661
2,Central,2020,3,138683.913361,539
3,Central,2020,4,141267.759734,507
4,Central,2020,5,140587.992559,581


## Join Baseline to Affected Counties Only

In [8]:
# Keep only storm-affected county-months for the analysis dataset
affected_panel = panel[panel['affected']].copy()

# Join regional baseline
affected_panel = affected_panel.merge(
    baseline,
    on=['climate_region', 'year', 'month'],
    how='left'
)

# Compute ZHVI deviation from baseline (positive = above baseline)
affected_panel['zhvi_vs_baseline'] = affected_panel['zhvi'] - affected_panel['baseline_zhvi']

print(f'Affected panel shape: {affected_panel.shape}')
print(f'Rows missing baseline: {affected_panel["baseline_zhvi"].isnull().sum()}')
affected_panel[['stcofips', 'year', 'month', 'climate_region', 'zhvi', 'baseline_zhvi', 'zhvi_vs_baseline', 'baseline_n_counties']].head(10)

Affected panel shape: (14166, 23)
Rows missing baseline: 0


,stcofips,year,month,climate_region,zhvi,baseline_zhvi,zhvi_vs_baseline,baseline_n_counties
0,48439,2020,1,South,2.296548e+05,150291.111115,79363.662121,483
1,48029,2020,1,South,1.973883e+05,150291.111115,47097.154973,483
2,06085,2020,1,West,1.151901e+06,372461.946201,779438.874365,85
3,06001,2020,1,West,8.497585e+05,372461.946201,477296.523499,85
4,48453,2020,1,South,3.730264e+05,150291.111115,222735.323333,483
5,48121,2020,1,South,3.135447e+05,150291.111115,163253.604293,483
6,06075,2020,1,West,1.303939e+06,372461.946201,931477.261451,85
7,06081,2020,1,West,1.324470e+06,372461.946201,952008.429313,85
8,40143,2020,1,South,1.592896e+05,150291.111115,8998.445974,483
9,48491,2020,1,South,3.011508e+05,150291.111115,150859.686397,483


## Validate

In [9]:
assert affected_panel.duplicated(['stcofips', 'year', 'month']).sum() == 0, 'Duplicate county-month rows'
assert affected_panel['baseline_zhvi'].isnull().sum() == 0, 'Missing baseline — check region assignment'
assert affected_panel['stcofips'].str.len().eq(5).all(), 'FIPS not all 5 digits'
assert affected_panel['baseline_n_counties'].gt(0).all(), 'Empty baseline'

print('All assertions passed')
print(f'\nBaseline county counts by region (min across all months):')
print(
    affected_panel.groupby('climate_region')['baseline_n_counties']
    .min()
    .sort_values()
    .reset_index()
)

All assertions passed

Baseline county counts by region (min across all months):
       climate_region  baseline_n_counties
0                West                   81
1           Southwest                   94
2  West North Central                  155
3           Northwest                  158
4           Northeast                  187
5  East North Central                  226
6               South                  292
7           Southeast                  435
8             Central                  498


## Export

In [10]:
output_cols = [
    'stcofips', 'state_fips', 'county_fips', 'climate_region',
    'year', 'month',
    'zhvi', 'baseline_zhvi', 'zhvi_vs_baseline', 'baseline_n_counties'
]

out = affected_panel[output_cols].sort_values(['stcofips', 'year', 'month']).reset_index(drop=True)

out.to_pickle('../data/processed/zillow_panel.pkl')

print(f'Exported zillow_panel.pkl')
print(f'Shape: {out.shape}')
print(f'Columns: {out.columns.tolist()}')

Exported zillow_panel.pkl
Shape: (14166, 10)
Columns: ['stcofips', 'state_fips', 'county_fips', 'climate_region', 'year', 'month', 'zhvi', 'baseline_zhvi', 'zhvi_vs_baseline', 'baseline_n_counties']


In [11]:
baseline.to_pickle('../data/processed/baseline_lookup.pkl')
print(f'Exported baseline_lookup.pkl')

Exported baseline_lookup.pkl
